In [106]:
import pandas as pd
from datetime import timedelta
import numpy as np
import ta.momentum
import ta.trend
import ta.volatility


In [107]:

# Strategy No. 1: ExtremeSpike
class ExtremeSpike:
    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df.copy()
        self.df['t'] = pd.to_datetime(self.df['datetime'])
        self.df['GMT'] = self.df['t'] + timedelta(hours=2)
        self.MINOR_MIN_EXTREME_HEIGHT_ATRS = 2.0
        self.MAJOR_TO_MINOR_HEIGHT_RATIO = 2.5
        self.MINOR_MIN_EXTREME_WIDTH = 2
        self.MAJOR_MIN_EXTREME_WIDTH = 2
        self.RANGE_AVERAGING_PERIOD = 250
        self.LINE_MINOR = 'minor'
        self.LINE_MAJOR = 'major'
        self.LINE_SHADOW = 'shadow'
        self.LINE_STABLE = 'stable'
        self.NoRepaint = True
        self.LINE_VALUE_UP = 1.0
        self.LINE_VALUE_FLAT = 0.0
        self.LINE_VALUE_DOWN = -1.0
        self.df['ATR'] = self.df['high'] - self.df['low']
        self.df['ATR_SMA'] = self.df['ATR'].rolling(window=self.RANGE_AVERAGING_PERIOD).mean()
        self.df['MinorMinExtremeHeight'] = self.df['ATR_SMA'] * self.MINOR_MIN_EXTREME_HEIGHT_ATRS
        self.df['MajorMinExtremeHeight'] = self.df['MinorMinExtremeHeight'] * self.MAJOR_TO_MINOR_HEIGHT_RATIO
        self.df['line1'] = 0.0
        self.df['line2'] = 0.0
        self.df['line3'] = 0.0
        self.df['line4'] = 0.0
        self.df['line5'] = 0.0
        self.minor = {
            "low_extreme_price": self.df['low'].iloc[0],
            "hi_extreme_price": self.df['high'].iloc[0],
            "low_extreme_idx": 0,
            "hi_extreme_idx": 0,
            "extreme_mode": 0,
            "first_low": 0,
            "first_high": 0,
        }
        self.major = {
            "low_extreme_price": self.df['low'].iloc[0],
            "hi_extreme_price": self.df['high'].iloc[0],
            "low_extreme_idx": 0,
            "hi_extreme_idx": 0,
            "extreme_mode": 0,
            "first_low": 0,
            "first_high": 0,
        }

    def eraseExtreme(self, lineType, barIdx, value):
        drawShadow = (lineType == self.LINE_MAJOR) and (self.NoRepaint or (value == self.LINE_VALUE_UP and self.df['line1'].iloc[barIdx] != 0) or (value == self.LINE_VALUE_DOWN and self.df['line2'].iloc[barIdx] != 0))
        self.drawExtreme(lineType, barIdx, self.LINE_VALUE_FLAT)
        if drawShadow:
            self.draw(self.LINE_SHADOW, barIdx, value)

    def drawExtreme(self, lineType, barIdx, value):
        if not self.NoRepaint:
            self.draw(lineType, barIdx, value)
            self.drawStableLine(lineType, barIdx, value)

    def drawStableLine(self, lineType, barIdx, value):
        if lineType == self.LINE_MAJOR:
            return False
        self.draw(self.LINE_STABLE, barIdx, value)
        return True

    def draw(self, lineType, barIdx, value):
        if lineType == self.LINE_MAJOR:
            self.updateLine('line1', 'line2', barIdx, value)
        elif lineType == self.LINE_MINOR:
            self.updateLine('line5', 'line5', barIdx, value)
        elif lineType == self.LINE_SHADOW:
            self.updateLine('line3', 'line3', barIdx, value)
        elif lineType == self.LINE_STABLE:
            self.updateLine('line4', 'line4', barIdx, value)

    def updateLine(self, lineUp, lineDown, barIdx, value):
        if value in [self.LINE_VALUE_FLAT, self.LINE_VALUE_UP]:
            self.df.loc[barIdx, lineUp] = value
        if value in [self.LINE_VALUE_FLAT, self.LINE_VALUE_DOWN]:
            self.df.loc[barIdx, lineDown] = value

    def check_for_extremes(self, ex_dict, min_extreme_height, min_extreme_width, current_idx, low, high, lineType, df):
        signal = 0
        line_value = 0.0
        if ex_dict['extreme_mode'] > -1:
            if low < ex_dict['low_extreme_price']:
                if not ex_dict['first_low']:
                    self.eraseExtreme(lineType, ex_dict['low_extreme_idx'], self.LINE_VALUE_DOWN)
                ex_dict['low_extreme_price'] = low
                ex_dict['low_extreme_idx'] = current_idx
                ex_dict['first_low'] = False
            elif low > ex_dict['low_extreme_price']:
                self.drawExtreme(lineType, ex_dict['low_extreme_idx'], self.LINE_VALUE_DOWN)
                ex_dict['first_low'] = False
                if ((low - ex_dict['low_extreme_price']) >= min_extreme_height) and ((current_idx - ex_dict['low_extreme_idx']) >= min_extreme_width):
                    ex_dict['extreme_mode'] = -1
                    ex_dict['hi_extreme_price'] = high
                    ex_dict['hi_extreme_idx'] = current_idx
                    ex_dict['first_high'] = True
                    ex_dict['first_low'] = True
                    line_value = self.LINE_VALUE_DOWN
                    if self.NoRepaint:
                        self.draw(lineType, ex_dict['low_extreme_idx'], self.LINE_VALUE_DOWN)
                    self.drawStableLine(lineType, ex_dict['low_extreme_idx'], self.LINE_VALUE_FLAT)
        if ex_dict['extreme_mode'] < 1:
            if high > ex_dict['hi_extreme_price']:
                if not ex_dict['first_high']:
                    self.eraseExtreme(lineType, ex_dict['hi_extreme_idx'], self.LINE_VALUE_UP)
                ex_dict['hi_extreme_price'] = high
                ex_dict['hi_extreme_idx'] = current_idx
                ex_dict['first_high'] = False
            elif high < ex_dict['hi_extreme_price']:
                self.drawExtreme(lineType, ex_dict['hi_extreme_idx'], self.LINE_VALUE_UP)
                ex_dict['first_high'] = False
                if ((ex_dict['hi_extreme_price'] - low) >= min_extreme_height) and ((current_idx - ex_dict['hi_extreme_idx']) >= min_extreme_width):
                    ex_dict['extreme_mode'] = 1
                    ex_dict['low_extreme_price'] = low
                    ex_dict['low_extreme_idx'] = current_idx
                    ex_dict['first_high'] = True
                    ex_dict['first_low'] = True
                    line_value = self.LINE_VALUE_UP
                    if self.NoRepaint:
                        self.draw(lineType, ex_dict['hi_extreme_idx'], self.LINE_VALUE_UP)
                    self.drawStableLine(lineType, ex_dict['hi_extreme_idx'], self.LINE_VALUE_FLAT)

    def mainloop(self):
        for idx in range(1, len(self.df)):
            self.check_for_extremes(
                self.minor,
                self.df['MinorMinExtremeHeight'].iloc[idx], self.MINOR_MIN_EXTREME_WIDTH, idx, self.df['low'].iloc[idx], self.df['high'].iloc[idx], 'minor', self.df
            )
            self.check_for_extremes(
                self.major,
                self.df['MajorMinExtremeHeight'].iloc[idx], self.MAJOR_MIN_EXTREME_WIDTH, idx, self.df['low'].iloc[idx], self.df['high'].iloc[idx], 'major', self.df
            )
        return self.df


In [108]:

class TMIndicator:

    def __init__(self,df: pd.DataFrame) -> None:
        self.df = df.copy()
        # Define the parameters
        self.half_length = 10
        self.price_column = 'close'
        self.bands_deviations = 2.4
        self.koeff = 0.0001
        self.interpolate = True
        # Initialize buffers
        self.tm_buffer = np.zeros(len(self.df))
        self.up_buffer = np.zeros(len(self.df))
        self.dn_buffer = np.zeros(len(self.df))
        self.wu_buffer = np.zeros(len(self.df))
        self.wd_buffer = np.zeros(len(self.df))
        self.up_arrow = np.full(len(self.df), np.nan)
        self.dn_arrow = np.full(len(self.df), np.nan)

    # Calculate the TMA and bands
    def calculate_tma(self, half_length, price_column, bands_deviations, koeff):
        full_length = 2.0 * half_length + 1.0
        
        for i in range(len(self.df)):
            sum_val = (half_length + 1) * self.df[price_column].iloc[i]
            sumw = half_length + 1
            for j in range(1, half_length + 1):
                if i + j < len(self.df):
                    sum_val += (half_length - j + 1) * self.df[price_column].iloc[i + j]
                    sumw += (half_length - j + 1)
                if i - j >= 0:
                    sum_val += (half_length - j + 1) * self.df[price_column].iloc[i - j]
                    sumw += (half_length - j + 1)
            
            self.tm_buffer[i] = sum_val / sumw
            
            if i >= half_length:
                diff = self.df[price_column].iloc[i] - self.tm_buffer[i]
                if i == half_length:
                    self.wu_buffer[i] = np.power(diff, 2) if diff >= 0 else 0
                    self.wd_buffer[i] = np.power(diff, 2) if diff < 0 else 0
                else:
                    self.wu_buffer[i] = (self.wu_buffer[i-1] * (full_length - 1) + np.power(diff, 2)) / full_length if diff >= 0 else self.wu_buffer[i-1] * (full_length - 1) / full_length
                    self.wd_buffer[i] = (self.wd_buffer[i-1] * (full_length - 1) + np.power(diff, 2)) / full_length if diff < 0 else self.wd_buffer[i-1] * (full_length - 1) / full_length

                self.up_buffer[i] = self.tm_buffer[i] + bands_deviations * np.sqrt(self.wu_buffer[i])
                self.dn_buffer[i] = self.tm_buffer[i] - bands_deviations * np.sqrt(self.wd_buffer[i])
        
        return self.tm_buffer, self.up_buffer, self.dn_buffer

    def calculate(self):
        return self.calculate_tma(self.half_length, self.price_column, self.bands_deviations, self.koeff)
    def mainloop(self):
        # Generate arrows
        self.tm_buffer, self.up_buffer, self.dn_buffer = self.calculate()
        for i in range(1, len(self.df) - 1):
            if self.df['high'].iloc[i+1] > self.up_buffer[i+1] and self.df['close'].iloc[i+1] > self.df['open'].iloc[i+1] and self.df['close'].iloc[i] < self.df['open'].iloc[i]:
                self.up_arrow[i] = self.df['high'].iloc[i] + self.df['close'].rolling(window=20).mean().iloc[i] + self.koeff
            if self.df['low'].iloc[i+1] < self.dn_buffer[i+1] and self.df['close'].iloc[i+1] < self.df['open'].iloc[i+1] and self.df['close'].iloc[i] > self.df['open'].iloc[i]:
                self.dn_arrow[i] = self.df['low'].iloc[i] - self.df['close'].rolling(window=20).mean().iloc[i] - self.koeff
        # return self.df
        decimal_length = len(str(self.df['open'].iloc[0]).split('.')[1])
        
        # Use .loc to avoid the SettingWithCopyWarning
        self.df.loc[:, 'tm_buffer'] = np.round(self.tm_buffer, decimal_length)
        self.df.loc[:, 'up_buffer'] = np.round(self.up_buffer, decimal_length)
        self.df.loc[:, 'dn_buffer'] = np.round(self.dn_buffer, decimal_length)
        self.df.loc[:, 'up_arrow'] = np.round(self.up_arrow, decimal_length)
        self.df.loc[:, 'dn_arrow'] = np.round(self.dn_arrow, decimal_length)
        
        # Vectorized computation for SELL_TM and BUY_TM
        self.df.loc[:, 'SELL_TM'] = (self.df[['open', 'close']].min(axis=1) <= self.df['up_buffer']) & (self.df['up_buffer'] <= self.df[['open', 'close']].max(axis=1))
        self.df.loc[:, 'BUY_TM'] = (self.df[['open', 'close']].min(axis=1) <= self.df['dn_buffer']) & (self.df['dn_buffer'] <= self.df[['open', 'close']].max(axis=1))
        
        return self.df[['BUY_TM','SELL_TM']]


In [109]:

# # Strategy No. 3: IINWMARROWS
# class IINWMARROWS:
#     def __init__(self, df: pd.DataFrame) -> None:
#         self.df = df.copy()
#         self.wml = 9
#         self.wmaw = 54
#         self.emal = 15
#         self.df['wma1'] = ta.trend.wma_indicator(self.df['close'], self.wml)
#         self.df['wma2'] = ta.trend.wma_indicator(self.df['close'], self.wmaw)
#         self.df['ema'] = ta.trend.ema_indicator(self.df['close'], self.emal)
#         self.df['up'] = np.zeros(len(self.df))
#         self.df['dn'] = np.zeros(len(self.df))

#     def mainloop(self):
#         for idx in range(len(self.df)):
#             if idx > 0:
#                 if self.df['wma1'].iloc[idx] > self.df['wma2'].iloc[idx] and self.df['wma1'].iloc[idx - 1] <= self.df['wma2'].iloc[idx - 1]:
#                     self.df.at[idx, 'ma_up'] = 1
#                 elif self.df['wma1'].iloc[idx] < self.df['wma2'].iloc[idx] and self.df['wma1'].iloc[idx - 1] >= self.df['wma2'].iloc[idx - 1]:
#                     self.df.at[idx, 'ma_dn'] = -1
#         return self.df


In [110]:

# Strategy No. 4: SuperArrow
class SuperArrow:
    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df.copy()
        self.df['rsi'] = ta.momentum.RSIIndicator(self.df['close'], window=13).rsi()
        self.df['bbu'] = ta.volatility.BollingerBands(self.df['close']).bollinger_hband()
        self.df['bbl'] = ta.volatility.BollingerBands(self.df['close']).bollinger_lband()
        self.df['bull_power'] = self.df['high'] - ta.trend.ema_indicator(self.df['close'], window=13)
        self.df['bear_power'] = self.df['low'] - ta.trend.ema_indicator(self.df['close'], window=13)
        self.df['ema_fast'] = ta.trend.ema_indicator(self.df['close'], window=3)
        self.df['ema_slow'] = ta.trend.ema_indicator(self.df['close'], window=34)
        self.df['super_buy_signal'] = 0
        self.df['super_sell_signal'] = 0

    def mainloop(self):
        for i in range(1, len(self.df)):
            if self.df['close'].iloc[i] > self.df['ema_fast'].iloc[i] and self.df['close'].iloc[i] < self.df['ema_slow'].iloc[i] and self.df['rsi'].iloc[i] < 30:
                self.df.at[i, 'super_buy_signal'] = 1
            elif self.df['close'].iloc[i] < self.df['ema_fast'].iloc[i] and self.df['close'].iloc[i] > self.df['ema_slow'].iloc[i] and self.df['rsi'].iloc[i] > 70:
                self.df.at[i, 'super_sell_signal'] = -1
        return self.df



In [111]:

# Strategy No. 5: BinaryArrow
class BinaryArrow:
    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df
        self.SignalGap = 20
        self.BarsToCount = 500
        self.dist = 24
        self.Point = 0.00001  # Assuming Point is 0.00001 for Forex pairs, adjust as needed
        self.df['binary_buy_signal'] = np.nan
        self.df['binary_sell_signal'] = np.nan

    def iHighest(self, highs: pd.Series, period: int, shift: int) -> int:
        if shift < 0 or shift + period > len(highs):
            return np.nan
        return highs.iloc[shift:shift + period].idxmax()

    def iLowest(self, lows: pd.Series, period: int, shift: int) -> int:
        if shift < 0 or shift + period > len(lows):
            return np.nan
        return lows.iloc[shift:shift + period].idxmin()

    def mainloop(self) -> pd.DataFrame:
        high_series = self.df['high']
        low_series = self.df['low']
        for i in range(self.BarsToCount, len(self.df)):
            start_index = max(0, i - self.dist // 2)
            hhb = self.iHighest(high_series, self.dist, start_index)
            llb = self.iLowest(low_series, self.dist, start_index)
            if pd.notna(hhb) and i == hhb:
                self.df.at[i, 'sell_signal'] = high_series.loc[hhb] + self.SignalGap * self.Point
            if pd.notna(llb) and i == llb:
                self.df.at[i, 'buy_signal'] = low_series.loc[llb] - self.SignalGap * self.Point
        return self.df[['binary_buy_signal', 'binary_sell_signal']]


In [112]:

# Strategy No. 6: SuperSignal
class SuperSignal:
    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df
        self.DIST = 24
        self.SIGNAL_GAP = 4
        self.df['Point'] = 0.0001
        self.df['High'] = self.df[['open', 'close']].max(axis=1) + np.random.rand(len(df)) * 0.01
        self.df['Low'] = self.df[['open', 'close']].min(axis=1) - np.random.rand(len(df)) * 0.01
        self.df['BUY_SUPER'] = np.nan
        self.df['SELL_SUPER'] = np.nan
        self.flagval1 = 0
        self.flagval2 = 0

    def mainloop(self):
        
        
        for i in range(self.DIST // 2, len(self.df)):
            highest_idx = self.df['High'][i - self.DIST // 2: i + self.DIST // 2 + 1].idxmax()
            lowest_idx = self.df['Low'][i - self.DIST // 2: i + self.DIST // 2 + 1].idxmin()
            if i == highest_idx:
                if i == len(self.df) - 1 and self.flagval1 == 0:
                    self.flagval1 = 1
                    self.flagval2 = 0
                    self.df.at[i, 'BUY_SUPER'] = self.df['High'][highest_idx] + self.SIGNAL_GAP * self.df['Point'][i]
            if i == lowest_idx:
                if i == len(self.df) - 1 and self.flagval2 == 0:
                    self.flagval2 = 1
                    self.flagval1 = 0
                    self.df.at[i, 'SELL_SUPER'] = self.df['Low'][lowest_idx] - self.SIGNAL_GAP * self.df['Point'][i]
        return self.df[['BUY_SUPER', 'SELL_SUPER']]


In [113]:

# Strategy No. 7: SuperSignalV3
class SuperSignalV3:
    def __init__(self, df: pd.DataFrame, dist1: int = 14, dist2: int = 21) -> None:
        self.dist1 = dist1
        self.dist2 = dist2
        self.df = df
        self._calculate_indicators()
        self._initialize_signals()

    def _calculate_indicators(self):
        self.df['HHB1'] = self.df['high'].rolling(window=self.dist1, center=True).max()
        self.df['LLB1'] = self.df['low'].rolling(window=self.dist1, center=True).min()
        self.df['HHB'] = self.df['high'].rolling(window=self.dist2, center=True).max()
        self.df['LLB'] = self.df['low'].rolling(window=self.dist2, center=True).min()
        self.df['ATR'] = ta.volatility.average_true_range(self.df['high'], self.df['low'], self.df['close'], window=50)

    def _initialize_signals(self):
        self.df['B1'] = np.nan
        self.df['B2'] = np.nan
        self.df['B3'] = np.nan
        self.df['B4'] = np.nan
        self.df['STRONG_BUY'] = np.nan
        self.df['STRONG_SELL'] = np.nan
        self.df['PARTIAL_BUY'] = np.nan
        self.df['PARTIAL_SELL'] = np.nan

    def calculateB(self):
        self.df['B1'] = np.where(self.df['high'] == self.df['HHB'], self.df['high'] + self.df['ATR'], np.nan)
        self.df['B2'] = np.where(self.df['low'] == self.df['LLB'], self.df['low'] - self.df['ATR'], np.nan)
        self.df['B3'] = np.where(self.df['high'] == self.df['HHB1'], self.df['high'] + self.df['ATR'] / 2, np.nan)
        self.df['B4'] = np.where(self.df['low'] == self.df['LLB1'], self.df['low'] - self.df['ATR'] / 2, np.nan)

    def mainloop(self):
        self.calculateB()
        conditions = [
            (self.df['B1'].notna() & self.df['B3'].notna(), 'STRONG_SELL', 2),
            (self.df['B1'].notna() & self.df['B3'].isna(), 'PARTIAL_SELL', 4),
            (self.df['B2'].notna() & self.df['B4'].notna(), 'STRONG_BUY', 1),
            (self.df['B2'].notna() & self.df['B4'].isna(), 'PARTIAL_BUY', 3)
        ]
        for condition, column, value in conditions:
            self.df.loc[condition, column] = value
        return self.df[['STRONG_BUY', 'STRONG_SELL', 'PARTIAL_BUY', 'PARTIAL_SELL']]


In [114]:
from datetime import timedelta
# Load your DataFrame
dataframe = pd.read_csv('common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')

# Initialize each strategy or indicator
extreme_spk = ExtremeSpike(dataframe)
tm_ind = TMIndicator(dataframe)
# imm_arrow = IINWMARROWS(dataframe)
super_arrow = SuperArrow(dataframe)
binary_arrow = BinaryArrow(dataframe)
super_signal = SuperSignal(dataframe)
super_signal_v3 = SuperSignalV3(dataframe)

# Run each strategy and concatenate the results
dfs = [
    extreme_spk.mainloop()[['datetime','open', 'close', 'high', 'low', 'line1', 'line2', 'line3', 'line4', 'line5']],
    tm_ind.mainloop()[['BUY_TM', 'SELL_TM']],
    # imm_arrow.mainloop()[['ma_up', 'ma_dn']],
    super_arrow.mainloop()[['super_buy_signal', 'super_sell_signal']],
    binary_arrow.mainloop()[['binary_buy_signal', 'binary_sell_signal']],
    super_signal.mainloop()[['BUY_SUPER', 'SELL_SUPER']],
    super_signal_v3.mainloop()[['STRONG_BUY', 'STRONG_SELL', 'PARTIAL_BUY', 'PARTIAL_SELL']]
]

# Merge or join the DataFrames if they share a common index or key
new_df = pd.concat(dfs, axis=1)

# Adjust timestamps and timezones
new_df['UTC'] = pd.to_datetime(new_df['datetime']) + timedelta(hours=5)
new_df['GMT'] = new_df['UTC'] + timedelta(hours=2)

# Select the desired columns for the final output
final_df = new_df[['datetime', 'UTC', 'GMT', 'open', 'high', 'low', 'close', 'line1', 'line2', 'line3', 'line4', 'line5',
                   'BUY_TM', 'SELL_TM', 'BUY_SUPER', 'SELL_SUPER', 'super_buy_signal', 'super_sell_signal',
                   'STRONG_BUY', 'STRONG_SELL', 'PARTIAL_BUY', 'PARTIAL_SELL']]

# Save the final DataFrame to CSV
output_file = 'common/MachineLearningModel/output/outputEurusd.csv'
final_df.to_csv(output_file, index=False)

print(f"Data successfully saved to {output_file}")


Data successfully saved to common/MachineLearningModel/output/outputEurusd.csv
